# 📅 Day 06: Custom Structures — Combining Data Structures

---

## The Art of Design Problems

Some problems need data structures that don't exist out of the box. You build them by **combining** existing ones. This is what "Design" problems test: can you identify which 2-3 building blocks to glue together?

The most famous example: **two heaps** for running median. Neither a sorted array, nor a single heap, nor a BST gives you O(log n) insert + O(1) median. But two heaps together do.

```
╔═══════════════════════════════════════════════════════════════════════╗
║  CUSTOM STRUCTURE = Combine 2+ structures + glue logic             ║
║                                                                     ║
║  Two Heaps:  max-heap (small half) + min-heap (large half)        ║
║  Lazy Heap:  heap + hashmap for stale entry detection              ║
║  OrderedSet: BST/sorted list + hashmap for O(1) lookup             ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🎯 Problem 1: Find Median from Data Stream (LC #295)

Add numbers one at a time, get median at any point.

### Why Two Heaps?

The median splits data into a **smaller half** and a **larger half**. If we maintain:
- A **max-heap** for the smaller half (so we can peek at the largest of the smalls)
- A **min-heap** for the larger half (so we can peek at the smallest of the larges)
- Keep them balanced (size difference ≤ 1)

Then the median is just the top of one or both heaps!

```
  Stream: 5, 2, 8, 1, 4
  
  After 5:    small=[5]  large=[]     → median = 5
  After 2:    small=[2]  large=[5]    → median = (2+5)/2 = 3.5
  After 8:    small=[5,2] large=[8]   → median = 5
  After 1:    small=[2,1] large=[5,8] → median = (2+5)/2 = 3.5
  After 4:    small=[4,2,1] large=[5,8] → median = 4
  
  (small is max-heap, so top is always the largest in the small half)
```

In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.small = []  # Max-heap (negate values since Python has min-heap)
        self.large = []  # Min-heap
    
    def addNum(self, num):
        # Always add to small first
        heapq.heappush(self.small, -num)
        
        # Ensure max of small ≤ min of large
        if self.large and -self.small[0] > self.large[0]:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        
        # Balance sizes: small can have at most 1 more than large
        if len(self.small) > len(self.large) + 1:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        elif len(self.large) > len(self.small):
            val = heapq.heappop(self.large)
            heapq.heappush(self.small, -val)
    
    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]  # Odd count → small has extra
        return (-self.small[0] + self.large[0]) / 2  # Even count → average

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
print(mf.findMedian())   # 1.5
mf.addNum(3)
print(mf.findMedian())   # 2.0

---

## 🎯 Problem 2: Design a Number Container (LC #2349)

Store numbers at indices. Find the smallest index containing a given number.

### The Lazy Deletion Pattern

When a value at an index changes, we don't remove it from the old heap — that would be O(n). Instead, we just add the new entry and **lazily clean up** stale entries when we access them.

```
  change(1, 10) → idx_to_num: {1:10}, heaps: {10: [1]}
  change(3, 10) → idx_to_num: {1:10, 3:10}, heaps: {10: [1,3]}
  find(10)      → peek heap for 10 → min index = 1 ✓
  change(1, 20) → idx_to_num: {1:20, 3:10}, heaps: {10: [1,3], 20: [1]}
  find(10)      → peek heap → index 1, but idx_to_num[1]=20≠10 → pop → index 3 ✓
```

In [ ]:
from collections import defaultdict
import heapq

class NumberContainers:
    def __init__(self):
        self.idx_to_num = {}                  # Current value at each index
        self.num_to_idxs = defaultdict(list)  # Min-heap of indices per value
    
    def change(self, index, number):
        self.idx_to_num[index] = number
        heapq.heappush(self.num_to_idxs[number], index)
    
    def find(self, number):
        heap = self.num_to_idxs[number]
        # Lazy deletion: pop stale entries where index no longer has this number
        while heap and self.idx_to_num.get(heap[0]) != number:
            heapq.heappop(heap)
        return heap[0] if heap else -1

nc = NumberContainers()
nc.change(1, 10)
nc.change(3, 10)
print(nc.find(10))  # 1 (smallest index with value 10)
nc.change(1, 20)    # Index 1 now has 20, not 10
print(nc.find(10))  # 3 (1 is stale, gets lazily popped)

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  CUSTOM STRUCTURE PATTERNS                                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  Running median           → Two heaps (max + min)                  ║
║  Stale value cleanup      → Lazy deletion (check validity on pop) ║
║  O(1) insert + O(1) rand  → Array + HashMap (LC #380)             ║
║  Sliding window median    → Two heaps + lazy delete               ║
║  Stream as intervals      → SortedDict + merge neighbors          ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 295 | Find Median from Data Stream | Hard | Two heaps (max + min) |
| 2349 | Design a Number Container | Medium | HashMap + lazy heap |
| 480 | Sliding Window Median | Hard | Two heaps with lazy deletion |
| 352 | Data Stream as Disjoint Intervals | Hard | SortedDict + merge |
| 380 | Insert Delete GetRandom O(1) | Medium | Array + HashMap |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Running median = two heaps (small max-heap + large min-heap)  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Lazy deletion: mark invalid, clean up when accessed            │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Custom structures = identify 2+ building blocks + glue logic  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Design problems: think about which ops need to be O(1)/O(logn)│
└──────────────────────────────────────────────────────────────────────┘
```

---

## 🎉 Chapter 13 Complete! DSA Quick Prep Notes — ALL DONE!

You now have notes covering all 13 chapters:
1. Arrays & Strings
2. Linked Lists + Strings
3. Stacks & Queues
4. Trees
5. Graphs
6. Dynamic Programming
7. Recursion & Backtracking
8. Greedy
9. Bit Manipulation
10. Math
11. Sorting & Searching
12. Design
13. Advanced Structures

**Go solve problems. Read the concept, understand the pattern, then grind.**